In [ ]:
import point_milling_frontend as pmf
import CylinderCollisionDetection as ccd
import numpy as np
import trimesh
import plotly.graph_objects as go

First, create the machining object

In [ ]:
# initialize a seed for reproducibility
np.random.seed(2)

# Control points of the bicubic bezier patch representing the machining surface
Q = 0.1*np.array([[0,2,-2,0], [1,0, 0, 1], [-1,1, 0.5, 1], [0,0,0,0]])
mat_Q = Q+Q.T
# Rotation function matrix
# In this case, we want the tool to go 'against the grain' of the surface, hence the rotation value is set to constant 0.5*Pi
mat_phi = 0.5*np.pi*np.ones((4,4))
# Tilt function matrix
# Let's make it random but relatively flat so that no crazy behaviors happen
# The values are in the range [0, 0.5*Pi], so we want the tilt values to be around 0.25*Pi
mat_theta = 0.25*np.pi + 0.1*(np.random.rand(4,4))
# The G_function is going to produce straight lines as levelsets
mat_G = np.array(
    [[0, 0, 0, 0],
     [1, 1, 1, 1],
     [2, 2, 2, 2],
     [3, 3, 3, 3]]
)

In [ ]:
real_surface_side_length = 100 # The side length of the surface is 100mm

# The tool radius is going to be 2cm
real_tool_radius = 20 # in mm
real_machining_tolerance =  0.05# 50 microns in mm
tool_radius = real_tool_radius / real_surface_side_length
machining_tolerance = real_machining_tolerance / real_surface_side_length
# offset distance m
m = 0
number_of_paths = 3
h = 0.01 # 1% of the width of the surface
surfaces_resolution = 100 # 100x100 points per surface
# Create the MachiningParameters object
machining_object = pmf.MachiningParameters(
                        R = tool_radius,
                        m = m, mat_Q = mat_Q,
                        matrices = [mat_G, mat_phi, mat_theta],
                        number_of_paths = number_of_paths,
                        h = h,
                        surfaces_resolution = surfaces_resolution,
                        machining_tolerance = machining_tolerance,
                        shank_length = 1+np.sqrt(5),
                        n_shanks = 10
                    )

Create now the shanks to test

In [ ]:
n_shanks = 5
set_of_shanks = machining_object.discrete_shanks_vectorized(n_shanks = n_shanks)

We can now create the collision detection object

In [ ]:
mesh, scene = pmf.pickable_to_mesh([machining_object.offset_vertices, machining_object.offset_triangles])
shanksObject = ccd.MultipleCylinders(
                set_of_cylinders = set_of_shanks,
                mesh=mesh,
                scene = scene,
                R = machining_object.R)

iteracion, information_matrix = shanksObject.collision_detection_no_gaps()

In [ ]:
information_matrix?

In [ ]:
# lets show the behaviour of all of this
fig = go.Figure()
# add the mesh
fig.add_mesh3d(
    x=machining_object.offset_vertices[:,0],
    y=machining_object.offset_vertices[:,1],
    z=machining_object.offset_vertices[:,2],
    i=machining_object.offset_triangles[:,0],
    j=machining_object.offset_triangles[:,1],
    k=machining_object.offset_triangles[:,2],
    color='lightgrey',
    opacity=0.5,
    name='Machining Surface'
)
# add the shanks
bottom_shank_points = set_of_shanks[:, 0]
top_shank_points = set_of_shanks[:, 1]

xs, ys, zs = [], [], []
for A, B in zip(bottom_shank_points, top_shank_points):
    xs.extend([A[0], B[0], None])
    ys.extend([A[1], B[1], None])
    zs.extend([A[2], B[2], None])

trace = go.Scatter3d(
    x=xs,
    y=ys,
    z=zs,
    mode="lines",
    line=dict(
        color="black",
        width=4
    ),
    name='Shanks Medial Axis',
    showlegend=True
)
fig.add_trace(trace)
points_in_cylinder = np.concatenate(information_matrix.points_in_cylinder)
footpoints = np.concatenate(information_matrix.footpoints)
# draw the line between the computed points int he shanks and their footpoints
xs, ys, zs = [], [], []
for A, B in zip(points_in_cylinder, footpoints):
    xs.extend([A[0], B[0], None])
    ys.extend([A[1], B[1], None])
    zs.extend([A[2], B[2], None])

trace = go.Scatter3d(
    x=xs,
    y=ys,
    z=zs,
    mode="lines",
    line=dict(
        color="blue",
        width=4,
    ),
    name='Lines Between Points in Cylinder and Footpoints',
    showlegend=True
)
fig.add_trace(trace)

#draw the safe sphere around the points in the cylinder with their corresponding safe_distance
# the sphere is drawn as an icosahedron from trimesh

for point, safe_distance in zip(points_in_cylinder, np.concatenate(information_matrix.safe_distances)):
    sphere_mesh = trimesh.creation.icosphere(subdivisions=3, radius=safe_distance)
    sphere_mesh.apply_translation(point)
    fig.add_mesh3d(
        x=sphere_mesh.vertices[:,0],
        y=sphere_mesh.vertices[:,1],
        z=sphere_mesh.vertices[:,2],
        i=sphere_mesh.faces[:,0],
        j=sphere_mesh.faces[:,1],
        k=sphere_mesh.faces[:,2],
        color='red',
        opacity=0.3,
        name='Safe Sphere',
        showlegend=False
    )



fig.update_layout(
    showlegend = True,
    scene=dict(
        aspectmode='data'),
        width = 900,
        height = 750
    )
fig.show()
